<a href="https://colab.research.google.com/github/sbeeredd04/sandbox/blob/main/token-opt/notebooks/clip_opt_imagefolder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLIP-Guided Test-Time Optimization with ImageFolder (XQ-GAN)

This notebook demonstrates CLIP-guided image editing and token interpretability using ImageFolder's XQ-GAN tokenizer with multi-scale product quantization.

In [1]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

In [2]:
import sys
if IN_COLAB:
    !git clone -q https://github.com/sbeeredd04/sandbox.git
    !pip install -q --progress-bar off jaxtyping open_clip_torch omegaconf timm
    sys.path.insert(0, "sandbox/token-opt")
    sys.path.insert(0, "sandbox/ImageFolder")
else:
    sys.path.insert(0, "..")
    sys.path.insert(0, "../../ImageFolder")

In [3]:
import os
# Set this environment for deterministic execution
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [4]:
import torch
# Enable for deterministic algorithms
torch.use_deterministic_algorithms(True, warn_only=False)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from pathlib import Path
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as v2
import torchvision.transforms.v2.functional as tvf
from torchvision.datasets import ImageNet
from einops import rearrange

/home/sbeeredd/miniconda3/envs/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from tto.test_time_opt import (
    TestTimeOpt,
    TestTimeOptConfig,
    CLIPObjective,
)

/home/sbeeredd/sandbox/continuous_tokenizer/modelling/modules/timm_vit/vision_transformer.py:2186: UserWarning: Overwriting vit_tiny_patch16_224 in registry with modelling.modules.timm_vit.vision_transformer.vit_tiny_patch16_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/home/sbeeredd/sandbox/continuous_tokenizer/modelling/modules/timm_vit/vision_transformer.py:2205: UserWarning: Overwriting vit_tiny_patch16_384 in registry with modelling.modules.timm_vit.vision_transformer.vit_tiny_patch16_384. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/home/sbeeredd/sandbox/continuous_tokenizer/modelling/modules/timm_vit/vision_transformer.py:2214: UserWarning: Overwriting vit_small_patch32_224 in registry with modelling.modules.timm_vit.vision_transformer.vit_small_patch32_224. This is because the name being registered conf

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Utils

In [7]:
def load_img(path, device=None):
    if IN_COLAB:
        path = Path("sandbox/token-opt/notebooks") / Path(path)
    img = (1. / 255.) * torch.from_numpy(
        np.array(Image.open(path)).astype(np.float32)
    ).permute(2, 0, 1)
    img = tvf.resize(img, 256)
    img = tvf.center_crop(img, 256)
    img = img.unsqueeze(0)
    if device is not None:
        img = img.to(device)
    return img

def display_image(*tensors):
    tensors = [255. * t.squeeze() for t in tensors]
    img = Image.fromarray(rearrange(
        tensors, "b c h w -> h (b w) c"
    ).to("cpu", dtype=torch.uint8).numpy())
    display(img)

def opt_callback(info):
    if info.i % 50 == 0:
        print(f"i = {info.i}")
        print("  CLIP score =", "\t".join(
            map(lambda l: f"{-l:.3f}", info.loss))
        )
        imgs = tto.decode(info.tokens).clamp(0., 1.)
        display_image(*imgs)

# Set up the objective function

In [8]:
# Use CLIP similarity maximization objective
objective = CLIPObjective(num_augmentations=8, cfg_scale=1.2)

# Set prompt
objective.prompt = [
    "a photo of a tiger",
    "a photo of a husky",
    "a photo of a sparrow",
]

# Optionally set a negative prompt
# Note: also need to set cfg_scale > 1 in CLIPObjective if using this!
objective.neg_prompt = "bad, low-res, unnatural"

# Configure test time optimization with ImageFolder

In [ ]:
tto_config = TestTimeOptConfig(
    # Use ImageFolder XQ-GAN tokenizer
    # Format: "imagefolder:MODEL_NAME" (auto-downloads checkpoint)
    # Available models: MSVR10P2-4096, MSVR10P2-8192, MSVR10P2-16384
    titok_checkpoint="imagefolder:MSVR10P2-4096",
    
    # Optimize in continuous space (before quantization)
    optimize_post_quantization_tokens=False,
    
    # VAE sampling (deterministic for reproducibility)
    vae_deterministic_sampling=True,
    
    # Optimization parameters
    num_iter=301,
    ema_decay=0.98,
    lr=1e-1,
    enable_amp=True,
    reg_weight=0.025,
    reg_type="seed",
)
tto = TestTimeOpt(tto_config, objective).to(device)

Using config for MSVR10P2-4096: {'codebook_size': 4096, 'codebook_embed_dim': 32, 'codebook_l2_norm': True, 'enc_type': 'dinov2', 'dec_type': 'dinov2', 'encoder_model': 'vit_base_patch14_dinov2.lvd142m', 'decoder_model': 'vit_base_patch14_dinov2.lvd142m', 'num_latent_tokens': 121, 'product_quant': 2, 'v_patch_nums': [1, 1, 2, 3, 3, 4, 5, 6, 8, 11], 'abs_pos_embed': True, 'share_quant_resi': 4, 'codebook_drop': 0.1, 'half_sem': True, 'start_drop': 3, 'hf_url': 'https://huggingface.co/qiuk6/XQ-GAN/resolve/main/MSVR10P2-4096/best_ckpt.pt'}
⚠ Cached checkpoint appears corrupted, re-downloading...


# Load seed images

In [ ]:
# Load seed image
img = torch.cat([
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00010240.png", device),
], dim=0)

# Alternatively, initialize directly from given tokens (e.g. randomly
# sampled), but this is disabled when setting `seed_tokens = None`.
seed_tokens = None

# Run Optimization

In [ ]:
print("Seed")
display_image(*img)

# Run optimization
torch.manual_seed(0)
img_opt = tto(
    seed=img if seed_tokens is None else None,
    seed_tokens=seed_tokens,
    callback=opt_callback
)

# ImageFolder Token Swapping Experiment

This section demonstrates token swapping between two images using ImageFolder's XQ-GAN model with product quantization (2 quantizers: semantic + detail).

ImageFolder uses a spatial 11×11 grid of tokens (121 tokens total) with multi-scale representation.

In [ ]:
from tto.imagefolder_wrapper import load_imagefolder_model

# Load ImageFolder model
imagefolder_model = load_imagefolder_model('MSVR10P2-4096')
imagefolder_model.eval()
imagefolder_model.to(device)

In [ ]:
def encode_tokens_imagefolder(model, img, quantize=False):
    """Encode image to tokens using ImageFolder model"""
    with torch.no_grad():
        # Encode: (b, c, h, w) -> (b, d, h, w)
        h = model.encoder(img)
        
        if quantize:
            # Quantize tokens
            h, _ = model.quantize(h)
        
        return h


def decode_tokens_imagefolder(model, tokens, quantize=False):
    """Decode tokens to image using ImageFolder model"""
    with torch.no_grad():
        # If not already quantized, quantize before decoding
        if not quantize:
            tokens, _ = model.quantize(tokens)
        
        # Decode: (b, d, h, w) -> (b, c, h, w)
        img = model.decode(tokens)
        return img


def swap_single_spatial_token(base_tokens, source_tokens, i, j):
    base_tokens[: , :, i, j] = source_tokens[:, :, i, j]
    return base_tokens

In [ ]:
# Set whether to quantize tokens immediately
quantize_tokens = False

# Load two images for token swapping
img1 = load_img("ILSVRC2012_val_00008636.png", device)
img2 = load_img("ILSVRC2012_val_00010240.png", device)

# Encode to tokens
tokens1 = encode_tokens_imagefolder(imagefolder_model, img1, quantize=quantize_tokens)
tokens2 = encode_tokens_imagefolder(imagefolder_model, img2, quantize=quantize_tokens)

print(f"Tokens shape: {tokens1.shape}")  # (1, d, h, w)
b, d, h, w = tokens1.shape
print(f"Spatial grid: {h}×{w} = {h*w} tokens")
print(f"Token dimension: {d}")

# Display original images
print("\n" + "="*60)
print("Original Images:")
display_image(img1, img2)
print("="*60)

# Decode to verify reconstruction
recon1 = decode_tokens_imagefolder(imagefolder_model, tokens1, quantize=quantize_tokens)
print("\nReconstructed Image 1:")
display_image(recon1)

## Token Swapping Loop

Now we'll swap each spatial token location one at a time and visualize the results. ImageFolder uses a 11×11 spatial grid (121 tokens).

In [ ]:
import imageio
from pathlib import Path

def save_img_tensor(tensor, path):
    """Save tensor as image file"""
    tensor = tensor.squeeze().clamp(0., 1.) * 255.
    img_array = tensor.permute(1, 2, 0).cpu().to(dtype=torch.uint8).numpy()
    img = Image.fromarray(img_array)
    img.save(path)
    return img_array


def concat_images(img1, img2):
    """Concatenate two images horizontally"""
    img1_np = img1.squeeze().clamp(0., 1.) * 255.
    img1_np = img1_np.permute(1, 2, 0).cpu().to(dtype=torch.uint8).numpy()

    img2_np = img2.squeeze().clamp(0., 1.) * 255.
    img2_np = img2_np.permute(1, 2, 0).cpu().to(dtype=torch.uint8).numpy()

    concat_images = np.concatenate([img1_np, img2_np], axis=1)
    return concat_images


# Create output directory
output_dir = Path("imagefolder_token_swap_frames")
output_dir.mkdir(exist_ok=True)

# Save original images
save_img_tensor(img1, output_dir / "original_img1.png")
save_img_tensor(img2, output_dir / "original_img2.png")

# Get spatial dimensions
b, d, h, w = tokens1.shape
num_tokens = h * w
print(f"Number of spatial tokens to swap: {num_tokens} ({h}×{w})")
print(f"{'='*60}\n")

# Collect frames for GIF
frames = []

print("Swapping tokens...")

# Swap each spatial location ONE AT A TIME
token_count = 0
for i in range(h):
    for j in range(w):
        token_count += 1
        print(f"Token {token_count}/{num_tokens} (position {i},{j})...", end="\r")
        
        # Pair 1: Start with ALL img1 tokens, swap ONLY position (i,j) from img2
        pair1_swapped = swap_single_spatial_token(tokens1, tokens2, i, j)
        
        # Pair 2: Start with ALL img2 tokens, swap ONLY position (i,j) from img1
        pair2_swapped = swap_single_spatial_token(tokens2, tokens1, i, j)
        
        # Decode both pairs
        img_pair1 = decode_tokens_imagefolder(imagefolder_model, pair1_swapped, quantize=quantize_tokens)
        img_pair2 = decode_tokens_imagefolder(imagefolder_model, pair2_swapped, quantize=quantize_tokens)
        
        # Create side-by-side images
        pair1_concat = concat_images(img1, img_pair1)
        pair2_concat = concat_images(img2, img_pair2)
        
        # Stack both pairs vertically
        combined_frame = np.concatenate([pair1_concat, pair2_concat], axis=0)
        
        # Save individual frames
        Image.fromarray(pair1_concat).save(output_dir / f"pair1_frame_{token_count:03d}.png")
        Image.fromarray(pair2_concat).save(output_dir / f"pair2_frame_{token_count:03d}.png")
        Image.fromarray(combined_frame).save(output_dir / f"combined_frame_{token_count:03d}.png")
        
        frames.append(combined_frame)

print(f"\n{'='*60}")
print(f"Token swapping complete! Swapped {num_tokens} tokens")
print(f"{'='*60}\n")

# Save as GIF
gif_path_combined = output_dir / "token_swap_combined.gif"
imageio.mimsave(gif_path_combined, frames, duration=100, loop=0)

print(f"GIF saved at {gif_path_combined}")

# Display the combined GIF
print("\nToken swap animation:")
from IPython.display import Image as IPImage
display(IPImage(filename=str(gif_path_combined)))

## Visualizing Product Quantization

ImageFolder uses product quantization with 2 separate quantizers. Let's visualize what each quantizer encodes.

In [ ]:
from math import sqrt

# Encode image
h = imagefolder_model.encoder(img1)  # (1, d, h, w)
b, c, l, _ = h.shape

print(f"Encoded features shape: {h.shape}")
print(f"Product quantization: {imagefolder_model.product_quant}x")

# Split for product quantization
h_list = h.chunk(chunks=imagefolder_model.product_quant, dim=2)

print(f"\nSplit into {len(h_list)} parts:")
for i, h_part in enumerate(h_list):
    print(f"  Part {i+1}: {h_part.shape}")

# Quantize each part separately
quant_list = []
for i, h_part in enumerate(h_list):
    h_reshaped = h_part.view(b, -1, int(sqrt(l // imagefolder_model.product_quant)), 
                             int(sqrt(l // imagefolder_model.product_quant)))
    quant, _, _, _, _ = imagefolder_model.model.quantizes[i].forward(
        h_reshaped, ret_usages=False, dropout=None
    )
    quant_list.append(quant)
    print(f"Quantizer {i+1} output shape: {quant.shape}")

# Decode each quantizer separately to see what they encode
print("\nDecoding each quantizer separately...")

# Quantizer 1 only (detail tokens)
quant_detail_only = torch.cat([quant_list[0], torch.zeros_like(quant_list[1])], dim=1)
img_detail = imagefolder_model.decode(quant_detail_only)

# Quantizer 2 only (semantic tokens)
quant_semantic_only = torch.cat([torch.zeros_like(quant_list[0]), quant_list[1]], dim=1)
img_semantic = imagefolder_model.decode(quant_semantic_only)

# Both quantizers (full reconstruction)
quant_both = torch.cat(quant_list, dim=1)
img_full = imagefolder_model.decode(quant_both)

print("\nVisualizing each quantizer's contribution:")
print("="*60)
print("Original Image:")
display_image(img1)
print("\nDetail Tokens Only (Quantizer 1):")
display_image(img_detail)
print("\nSemantic Tokens Only (Quantizer 2):")
display_image(img_semantic)
print("\nBoth Quantizers Combined:")
display_image(img_full)
print("="*60)

# Save visualizations
from torchvision.utils import save_image
save_image(img_detail, output_dir / "detail_tokens_only.png")
save_image(img_semantic, output_dir / "semantic_tokens_only.png")
save_image(img_full, output_dir / "both_quantizers.png")
print(f"\n✓ Saved visualizations to {output_dir}")

## Summary

**ImageFolder (XQ-GAN) Architecture:**
- **Tokens**: 121 spatial tokens (11×11 grid)
- **Product Quantization**: 2 quantizers (semantic + detail)
- **Multi-scale**: 10 levels of progressive refinement
- **Encoder**: DINOv2 ViT-Base with learnable tokens
- **Decoder**: DINOv2 ViT-Base

**Key Observations:**
1. **Spatial Structure**: Tokens maintain 2D spatial layout (11×11) unlike TiTok's 1D sequence
2. **Product Quantization**: Quantizer 1 captures fine details, Quantizer 2 captures semantic content
3. **Multi-scale**: Progressive refinement from coarse (1×1) to fine (11×11)
4. **Token Interpretability**: Swapping spatial locations shows localized effects

This architecture (Figure 3 from ImageFolder paper) is specifically designed for token interpretability and semantic/detail separation!